<a href="https://colab.research.google.com/github/CienciaDatosUdea/005_CCA_Estudiantes/blob/main/Sesiones/U4_S3_MessagePassing_parte1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Unidad 4 · Sesión 3, parte 1 — Del grafo al paso de mensajes

Cuaderno de ilustración de la primera parte de la clase (Hamilton, apertura del cap. 5 y secs. 5.1–5.2): el grafo $K_4-e$ y sus permutaciones, la fórmula de Euler, invariancia y equivarianza, la GNN básica y la normalización de la GCN. La segunda parte está en `U4_S3_MessagePassing_parte2.ipynb`.

Lo ejecutamos juntos celda por celda. Hay tres **ejercicios** para resolver en clase; su solución está en una celda oculta justo debajo (en Colab, haga clic en *Mostrar código*).

En Colab, PyTorch ya viene instalado; falta PyTorch Geometric (PyG), que usamos en la sección 6.

In [ ]:
!pip install -q torch_geometric

In [ ]:
import itertools
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import torch

Una función para dibujar todos los grafos con el mismo estilo:

In [ ]:
def dibujar(G, pos, ax, titulo=""):
    nx.draw_networkx(G, pos, ax=ax, node_color="#d9f2e1", edgecolors="#009b3a",
                     node_size=500, width=2, edge_color="0.4")
    ax.set_title(titulo)
    ax.margins(0.2)
    ax.set_aspect("equal")
    ax.axis("off")

## 1. Un grafo es quién está unido con quién

Usamos como ejemplo $K_4-e$: el grafo completo de cuatro nodos al que le falta una arista. Lo damos con su **lista de adyacencia**, que dice, para cada nodo, quiénes son sus vecinos. En la lista A los vecinos de cada nodo están escritos en el orden en que se encuentran al girar alrededor de él en sentido antihorario en el dibujo de abajo; ese orden lo usaremos en la sección 3.

In [ ]:
lista_A = {0: [1, 3], 1: [0, 2, 3], 2: [1, 3], 3: [0, 1, 2]}

G = nx.Graph()
G.add_nodes_from(range(4))
G.add_edges_from((u, v) for u in lista_A for v in lista_A[u])
print("aristas:", sorted(G.edges()))
print("grados:", dict(G.degree()))

Para dibujarlo se recorre la lista: se toma un nodo, se traza la arista hacia cada vecino con el que todavía no esté unido, y se sigue con el nodo siguiente hasta agotar las aristas. Cada panel añade una arista.

In [ ]:
pos_A = {0: (0, 1), 1: (-1, 0), 3: (1, 0), 2: (0, -1)}

orden = []
for u in lista_A:
    for v in lista_A[u]:
        if (v, u) not in orden:
            orden.append((u, v))

fig, axs = plt.subplots(1, len(orden), figsize=(13, 2.8))
for k, ax in enumerate(axs):
    H = nx.Graph(orden[:k + 1])
    H.add_nodes_from(G)
    dibujar(H, pos_A, ax, f"arista {orden[k]}")
plt.show()

## 2. Renombrar los nodos no cambia el grafo

Una permutación $\sigma$ cambia el nombre de los nodos: el nodo $i$ pasa a llamarse $\sigma(i)$. En la matriz de adyacencia eso es
$$A' = P A P^\top,\qquad P_{\sigma(i),\,i}=1 ,$$
y los atributos de los nodos se reordenan igual, $X' = PX$.

In [ ]:
def matriz_P(sigma):
    # P[sigma[i], i] = 1: el nodo i pasa a llamarse sigma[i]
    n = len(sigma)
    P = np.zeros((n, n), dtype=int)
    P[sigma, range(n)] = 1
    return P

El ejemplo de las diapositivas: intercambiar los nodos 2 y 3. La matriz cambia, pero el grafo no.

In [ ]:
A_A = nx.to_numpy_array(G, nodelist=range(4), dtype=int)
sigma23 = [0, 1, 3, 2]                     # intercambia los nodos 2 y 3
P23 = matriz_P(sigma23)
A_B = P23 @ A_A @ P23.T

print("A_A =\n", A_A)
print("A_B = P A_A P^T =\n", A_B)
print("¿mismo grafo?", nx.is_isomorphic(nx.from_numpy_array(A_A), nx.from_numpy_array(A_B)))

Con las $4! = 24$ permutaciones, ¿cuántas matrices distintas aparecen? Las permutaciones que dejan la matriz igual son los **automorfismos** del grafo; el número de matrices distintas es $n!/|\mathrm{Aut}|$.

In [ ]:
distintas = {}
automorfismos = 0
for sigma in itertools.permutations(range(4)):
    P = matriz_P(list(sigma))
    Ap = P @ A_A @ P.T
    distintas.setdefault(Ap.tobytes(), Ap)
    automorfismos += np.array_equal(Ap, A_A)

matrices = list(distintas.values())
print(len(matrices), "matrices distintas;  |Aut| =", automorfismos, "; 24 /", automorfismos, "=", 24 // automorfismos)

Las seis matrices dibujadas con los nodos fijos en las esquinas (0 arriba a la izquierda, 1 arriba a la derecha, 2 abajo a la derecha, 3 abajo a la izquierda). En los seis dibujos hay dos nodos de grado 2, dos de grado 3 y una sola arista que falta: es el mismo grafo con los nombres en otro lugar. El dibujo tiene un cruce cuando están las dos diagonales 0–2 y 1–3.

In [ ]:
pos_esq = {0: (0, 1), 1: (1, 1), 2: (1, 0), 3: (0, 0)}

fig, axs = plt.subplots(2, 6, figsize=(13, 4.6))
for k, Ap in enumerate(matrices):
    dibujar(nx.from_numpy_array(Ap), pos_esq, axs[0, k], f"matriz {k + 1}")
    axs[1, k].imshow(Ap, cmap="Greens", vmin=0, vmax=1.5)
    axs[1, k].set_xticks(range(4))
    axs[1, k].set_yticks(range(4))
plt.tight_layout()
plt.show()

## 3. El dibujo no es el grafo: caras, fórmula de Euler y género

Si en la lista de cada nodo los vecinos están en el orden en que se encuentran al girar alrededor de él (sentido antihorario), la lista fija una **inmersión**: una manera de dibujar el grafo sobre una superficie sin cruces. Las **caras** se recorren así: se llega al nodo $v$ desde $u$ y se sale hacia el vecino que sigue a $u$ en la lista de $v$ (volviendo al principio de la lista si $u$ es el último). Cada arista se recorre una vez en cada sentido.

Con el número de caras $F$, la fórmula de Euler (válida cuando cada cara es un disco)
$$V - L + F = 2 - 2g$$
da el **género** $g$ de la superficie: $g=0$ es la esfera o el plano, $g=1$ el toro.

In [ ]:
def caras(lista):
    siguiente = {v: {w: vec[(i + 1) % len(vec)] for i, w in enumerate(vec)}
                 for v, vec in lista.items()}
    libres = [(u, v) for u in lista for v in lista[u]]     # cada arista en los dos sentidos
    resultado = []
    while libres:
        u, v = libres[0]
        cara = []
        while (u, v) in libres:
            libres.remove((u, v))
            cara.append(u)
            u, v = v, siguiente[v][u]                      # al llegar a v, salir por el siguiente
        resultado.append(cara)
    return resultado


def genero(lista):
    V = len(lista)
    L = sum(len(vec) for vec in lista.values()) // 2
    F = len(caras(lista))
    return V, L, F, (2 - V + L - F) // 2

La lista A, que leímos del dibujo plano, y la lista B, que es la lista de $A_B$ con los vecinos en orden creciente:

In [ ]:
lista_B = {0: [1, 2], 1: [0, 2, 3], 2: [0, 1, 3], 3: [1, 2]}

for nombre, lista in [("A", lista_A), ("B", lista_B)]:
    V, L, F, g = genero(lista)
    print(f"lista {nombre}: caras {caras(lista)}")
    print(f"         V={V}, L={L}, F={F}  ->  g = {g}")

La lista A tiene tres caras (dos triángulos y la cara exterior): el plano. La lista B tiene una sola cara: Euler da $g=1$, el toro. Aquí está el dibujo sin cruces de la lista B en el toro, representado como un cuadrado cuyos lados opuestos están identificados: una arista que sale por un lado entra por el lado opuesto.

In [ ]:
segmentos = [((1.2, 2.6), (2.8, 1.4)),                              # 1-2
             ((1.2, 2.6), (0, 2.809)), ((4, 2.809), (2.9, 3.0)),    # 1-0, cruza el lado izquierdo
             ((2.8, 1.4), (2.9, 3.0)),                              # 2-0
             ((1.2, 2.6), (1.142, 4)), ((1.142, 0), (1.1, 1.0)),    # 1-3, cruza el lado superior
             ((2.8, 1.4), (1.1, 1.0))]                              # 2-3
nodos_toro = {0: (2.9, 3.0), 1: (1.2, 2.6), 2: (2.8, 1.4), 3: (1.1, 1.0)}

fig, ax = plt.subplots(figsize=(4.2, 4.2))
ax.plot([0, 4, 4, 0, 0], [0, 0, 4, 4, 0], color="0.6")
for x, y, s in [(2, 0, ">"), (2, 4, ">"), (0, 2, "^"), (4, 2, "^")]:
    ax.text(x, y, s, ha="center", va="center", color="0.5", fontsize=14)
for (x1, y1), (x2, y2) in segmentos:
    ax.plot([x1, x2], [y1, y2], color="0.4", lw=2)
for v, (x, y) in nodos_toro.items():
    ax.scatter(x, y, s=500, color="#d9f2e1", edgecolors="#009b3a", zorder=3)
    ax.text(x, y, v, ha="center", va="center", zorder=4)
ax.set_title("lista B en el toro")
ax.set_aspect("equal")
ax.axis("off")
plt.show()

Las dos listas describen el mismo grafo ($A_B = P A_A P^\top$), pero ordenadas fijan superficies distintas. Que la lista B viva en el toro es una propiedad de esa lista ordenada, no del grafo: el **género de un grafo** es el mínimo sobre todas sus inmersiones, y $K_4-e$ es plano.

### Ejercicio 1
Escriba las listas de adyacencia de las seis matrices de la sección 2, con los vecinos en orden creciente. Recorra las caras de cada una con `caras` y compruebe que dos dan el plano ($F=3$) y cuatro el toro ($F=1$). ¿Qué tienen en común los dibujos en las esquinas de las dos listas planas?

In [ ]:
#@title Solución del ejercicio 1
for k, Ap in enumerate(matrices, start=1):
    lista = {v: [int(w) for w in np.flatnonzero(Ap[v])] for v in range(4)}
    V, L, F, g = genero(lista)
    print(f"matriz {k}: {lista}   F = {F}, g = {g}")
# Planas: las matrices cuyo dibujo en las esquinas no tiene cruces, es decir,
# aquellas a las que les falta una diagonal (0-2 o 1-3).
# Las otras cuatro tienen las dos diagonales: su lista ordenada da una sola cara, el toro.

## 4. Fórmula de Euler con `nx.check_planarity`

NetworkX busca una inmersión plana (`check_planarity`) y, si la encuentra, devuelve su sistema de rotaciones: la lista ordenada de vecinos de cada nodo. Contamos sus caras con `traverse_face` y comprobamos que $V-L+F=2$ para grafos planos y conexos. $K_5$ no tiene inmersión plana.

In [ ]:
def caras_planas(G):
    es_plano, inmersion = nx.check_planarity(G)
    if not es_plano:
        return None
    vistas, F = set(), 0
    for u, v in inmersion.edges():
        if (u, v) not in vistas:
            inmersion.traverse_face(u, v, mark_half_edges=vistas)
            F += 1
    return F


grafos = {"C6": nx.cycle_graph(6), "K4": nx.complete_graph(4), "K4-e": G,
          "familias florentinas": nx.florentine_families_graph(), "K5": nx.complete_graph(5)}
for nombre, H in grafos.items():
    V, L, F = H.number_of_nodes(), H.number_of_edges(), caras_planas(H)
    if F is None:
        print(f"{nombre:21s} V={V:2d}  L={L:2d}         no es plano")
    else:
        print(f"{nombre:21s} V={V:2d}  L={L:2d}  F={F:2d}   V - L + F = {V - L + F}")

## 5. Por qué un MLP sobre la adyacencia falla

Un MLP recibe la matriz aplanada, $\mathrm{vec}(A)$, con 16 entradas, y trata la entrada $(i,j)$ siempre de la misma manera. Al renombrar los nodos la entrada cambia, y la salida también, aunque el grafo sea el mismo. Lo vemos con un MLP de pesos aleatorios.

In [ ]:
rng = np.random.default_rng(0)
W1, W2 = rng.normal(size=(16, 8)), rng.normal(size=(8, 1))


def mlp(A):
    return (np.tanh(A.reshape(-1) @ W1) @ W2).item()


print(f"MLP(A_A) = {mlp(A_A):.3f}    MLP(A_B) = {mlp(A_B):.3f}")

Una función **invariante** cumple $f(PAP^\top) = f(A)$: no depende de los nombres. El número de aristas $\tfrac12\mathbf 1^\top A\mathbf 1$, el número de triángulos $\operatorname{tr}(A^3)/6$ y el espectro lo son.

In [ ]:
def invariantes(A):
    return {"aristas": int(A.sum()) // 2,
            "triángulos": int(np.trace(A @ A @ A)) // 6,
            "espectro": np.round(np.linalg.eigvalsh(A), 3)}


print("A_A:", invariantes(A_A))
print("A_B:", invariantes(A_B))

Una función **equivariante** cumple $f(PAP^\top) = P f(A)$: da un valor por nodo, y al renombrar los nodos los valores se reordenan con ellos. El vector de grados $A\mathbf 1$ y la agregación $AX$ lo son.

In [ ]:
print("grados de A_A:      ", A_A.sum(1))
print("grados de A_B:      ", A_B.sum(1))
print("P · grados de A_A:  ", P23 @ A_A.sum(1))

X = rng.normal(size=(4, 3))
print("(P A P^T)(P X) = P (A X):", np.allclose(A_B @ (P23 @ X), P23 @ (A_A @ X)))

### Ejercicio 2
Elija tres permutaciones de los nodos de $K_4-e$ y dibuje, con los nodos en las esquinas, el grafo que resulta de cada una. Compruebe con $PAP^\top$ que es el mismo grafo, que el número de aristas y el espectro no cambian y que el vector de grados se permuta con $P$.

In [ ]:
#@title Solución del ejercicio 2
permutaciones = [[1, 0, 2, 3], [3, 0, 1, 2], [0, 2, 1, 3]]

fig, axs = plt.subplots(1, 3, figsize=(9, 3))
for sigma, ax in zip(permutaciones, axs):
    P = matriz_P(sigma)
    Ap = P @ A_A @ P.T
    dibujar(nx.from_numpy_array(Ap), pos_esq, ax, f"σ = {sigma}")
    print(f"σ = {sigma}:  mismo grafo {nx.is_isomorphic(G, nx.from_numpy_array(Ap))},"
          f"  aristas {Ap.sum() // 2},"
          f"  mismo espectro {np.allclose(np.linalg.eigvalsh(Ap), np.linalg.eigvalsh(A_A))},"
          f"  grados = P·grados {np.array_equal(Ap.sum(1), P @ A_A.sum(1))}")
plt.show()
# Si σ es un automorfismo (por ejemplo [2, 3, 0, 1]), la matriz no cambia.

## 6. La GNN básica (Hamilton §5.1)

Cada capa hace AGGREGATE (sumar los estados de los vecinos) y UPDATE (combinar con el propio estado):
$$\mathbf h_u^{(k)}=\sigma\Big(W_{\text{self}}^{(k)}\mathbf h_u^{(k-1)}+W_{\text{neigh}}^{(k)}\sum_{v\in\mathcal N(u)}\mathbf h_v^{(k-1)}+\mathbf b^{(k)}\Big),$$
y en forma matricial, con los estados como filas de $H$, $H^{(k)}=\sigma\big(AH^{(k-1)}W_{\text{neigh}}+H^{(k-1)}W_{\text{self}}\big)$.

La implementación no usa la matriz $A$ sino la lista de aristas `edge_index`, como en PyG: la fila 0 es el origen del mensaje (el vecino $v$) y la fila 1 el destino ($u$). `index_add_` suma cada mensaje en la fila de su destino.

In [ ]:
edge_index = torch.tensor([[v for u in lista_A for v in lista_A[u]],     # origen
                           [u for u in lista_A for v in lista_A[u]]])    # destino
print(edge_index)


def capa_gnn(H, edge_index, W_self, W_neigh, b):
    origen, destino = edge_index
    m = torch.zeros(H.shape[0], H.shape[1]).index_add_(0, destino, H[origen])   # AGGREGATE: suma
    return torch.tanh(H @ W_self + m @ W_neigh + b)                             # UPDATE

Aplicamos una capa con pesos aleatorios a atributos aleatorios y comprobamos que coincide con la forma matricial.

In [ ]:
torch.manual_seed(0)
H0 = torch.randn(4, 3)
W_self, W_neigh, b = torch.randn(3, 2), torch.randn(3, 2), torch.randn(2)

H1 = capa_gnn(H0, edge_index, W_self, W_neigh, b)
A_t = torch.tensor(A_A, dtype=torch.float32)
print(H1)
print("igual a la forma matricial:", torch.allclose(H1, torch.tanh(H0 @ W_self + A_t @ H0 @ W_neigh + b), atol=1e-6))

**Equivarianza.** Renombramos los nodos con $\sigma$ (intercambiar 2 y 3): los atributos pasan a $PH$ y cada arista $(v,u)$ a $(\sigma(v),\sigma(u))$. La salida de la capa debe ser la misma salida reordenada, $P H^{(1)}$; el MLP de la sección 5 no cumplía nada parecido.

In [ ]:
P_t = torch.tensor(P23, dtype=torch.float32)
sig = torch.tensor(sigma23)

H1_B = capa_gnn(P_t @ H0, sig[edge_index], W_self, W_neigh, b)
print("capa(PH, σ(aristas)) = P capa(H, aristas):", torch.allclose(H1_B, P_t @ H1, atol=1e-6))

**La misma capa en PyG.** `MessagePassing` separa los dos pasos: `message` dice qué envía cada vecino $j$ (aquí su estado $\mathbf x_j$), `aggr="add"` los suma y `forward` hace el UPDATE.

In [ ]:
from torch_geometric.nn import MessagePassing


class CapaBasica(MessagePassing):
    def __init__(self, W_self, W_neigh, b):
        super().__init__(aggr="add")              # AGGREGATE = suma
        self.W_self, self.W_neigh, self.b = W_self, W_neigh, b

    def message(self, x_j):                        # mensaje del vecino j: su estado
        return x_j

    def forward(self, x, edge_index):
        m = self.propagate(edge_index, x=x)        # m_u = suma de x_j sobre los vecinos de u
        return torch.tanh(x @ self.W_self + m @ self.W_neigh + self.b)   # UPDATE


capa = CapaBasica(W_self, W_neigh, b)
print("PyG da lo mismo que index_add_:", torch.allclose(capa(H0, edge_index), H1, atol=1e-6))

### Ejercicio 3
Cambie la agregación de suma a **media** (divida la suma por el grado de cada nodo) y compruebe que la capa sigue siendo equivariante. En PyG basta `aggr="mean"`.

In [ ]:
#@title Solución del ejercicio 3
def capa_gnn_media(H, edge_index, W_self, W_neigh, b):
    origen, destino = edge_index
    suma = torch.zeros(H.shape[0], H.shape[1]).index_add_(0, destino, H[origen])
    grado = torch.zeros(H.shape[0]).index_add_(0, destino, torch.ones(destino.shape[0]))
    m = suma / grado[:, None]
    return torch.tanh(H @ W_self + m @ W_neigh + b)


H1m = capa_gnn_media(H0, edge_index, W_self, W_neigh, b)
H1m_B = capa_gnn_media(P_t @ H0, sig[edge_index], W_self, W_neigh, b)
print("equivariante con la media:", torch.allclose(H1m_B, P_t @ H1m, atol=1e-6))
# La media de los vecinos no depende de sus nombres; el grado se permuta con los nodos.

## 7. Normalización: la GCN (Hamilton §5.2)

Con la suma, un nodo de grado alto recibe mensajes más grandes, y al apilar capas los estados crecen como $\lambda_{\max}(A)^k$. La GCN agrega con auto-lazos y normalización simétrica,
$$\hat A=\tilde D^{-1/2}\tilde A\,\tilde D^{-1/2},\qquad \tilde A=A+I,\quad \tilde D=\operatorname{diag}(\tilde A\mathbf 1),$$
cuyos autovalores están en $(-1,1]$.

In [ ]:
def gcn_normalizada(A):
    At = A + np.eye(len(A))
    d = At.sum(1)
    return At / np.sqrt(np.outer(d, d))


A_gcn = gcn_normalizada(A_A)
print("Â =\n", np.round(A_gcn, 3))
print("autovalores de A:", np.round(np.linalg.eigvalsh(A_A), 3))
print("autovalores de Â:", np.round(np.linalg.eigvalsh(A_gcn), 3))

Aplicamos $k$ veces cada matriz a un vector de unos (sin pesos ni no linealidad, para ver solo el efecto de la agregación):

In [ ]:
h = np.ones(4)
normas_A = [np.linalg.norm(np.linalg.matrix_power(A_A, k) @ h) for k in range(11)]
normas_gcn = [np.linalg.norm(np.linalg.matrix_power(A_gcn, k) @ h) for k in range(11)]

fig, ax = plt.subplots(figsize=(5, 3.2))
ax.semilogy(normas_A, "o-", label=r"suma: $A^k h$")
ax.semilogy(normas_gcn, "s-", label=r"GCN: $\hat A^k h$")
ax.set_xlabel("número de capas $k$")
ax.set_ylabel(r"$\|h^{(k)}\|$")
ax.legend()
plt.show()

## Resumen

- Un grafo es su lista de adyacencia; renombrar los nodos ($A\to PAP^\top$) da otra matriz, otra lista ordenada y a veces otra superficie, pero el mismo grafo.
- Un MLP sobre $\mathrm{vec}(A)$ depende de los nombres; las funciones útiles sobre grafos son invariantes o equivariantes.
- La capa de paso de mensajes (AGGREGATE con suma o media, UPDATE) es equivariante por construcción; la GCN normaliza la agregación para que los estados no crezcan.

En la parte 2 apilamos muchas de estas capas, leemos el grafo entero y armamos mini-lotes.